In [1]:
# Milestone 2 · Step 1 — get the project from GitHub and build the search index
!git clone -q https://github.com/jamalla/arabic-book-recommender-agent.git repo
import sys
sys.path.insert(0, "repo")

from bookrec.search import BookSearch, book_documents, normalize_arabic

engine = BookSearch("repo/data/catalog.csv")
print(engine.matrix.shape[0], "books ×", engine.matrix.shape[1], "character n-grams in the vocabulary")

200 books × 23846 character n-grams in the vocabulary


In [2]:
# Step 2 — normalization: spelling variants of the same word become identical
pairs = [("الإسلامية", "الاسلاميه"), ("رواية", "روايه"), ("مستشفى", "مستشفي"), ("القُرْآن", "القران")]
for a, b in pairs:
    na, nb = normalize_arabic(a), normalize_arabic(b)
    print(f"{a}  |  {b}   ->   {na}  |  {nb}   same: {na == nb}")

الإسلامية  |  الاسلاميه   ->   الاسلاميه  |  الاسلاميه   same: True
رواية  |  روايه   ->   روايه  |  روايه   same: True
مستشفى  |  مستشفي   ->   مستشفي  |  مستشفي   same: True
القُرْآن  |  القران   ->   القران  |  القران   same: True


In [3]:
# Step 3 — what the vectorizer "sees": character n-grams (2 to 4 letters, padded with spaces)
analyzer = engine.vectorizer.build_analyzer()
for a, b in [("رواية", "روايات"), ("تربية", "لتربية")]:
    ga, gb = set(analyzer(a)), set(analyzer(b))
    print(f"{a}: {len(ga)} n-grams | {b}: {len(gb)} n-grams | shared: {len(ga & gb)} -> {sorted(ga & gb)}")
print("\nA WORD-level index sees these as 4 unrelated words. The CHAR-level index sees large overlaps.")

رواية: 15 n-grams | روايات: 18 n-grams | shared: 9 -> [' ر', ' رو', ' روا', 'اي', 'رو', 'روا', 'رواي', 'وا', 'واي']
تربية: 15 n-grams | لتربية: 18 n-grams | shared: 12 -> ['بي', 'بية', 'بية ', 'ة ', 'تر', 'ترب', 'تربي', 'رب', 'ربي', 'ربية', 'ية', 'ية ']

A WORD-level index sees these as 4 unrelated words. The CHAR-level index sees large overlaps.


In [4]:
# Step 4 — word-level vs character-level TF-IDF on the same reader requests
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

docs = book_documents(engine.books)
word_vectorizer = TfidfVectorizer(analyzer="word")
word_matrix = word_vectorizer.fit_transform(docs)

def top3(vectorizer, matrix, query):
    scores = linear_kernel(vectorizer.transform([normalize_arabic(query)]), matrix).ravel()
    return [f"{engine.books.title[i][:45]}  ({scores[i]:.2f})" for i in scores.argsort()[::-1][:3]]

rows = []
for q in ["تربية الاطفال", "الحرب العالمية الثانية", "الذكاء الاصطناعي", "هاري بوتر"]:
    pairs = zip(top3(word_vectorizer, word_matrix, q), top3(engine.vectorizer, engine.matrix, q))
    for rank, (w, c) in enumerate(pairs, 1):
        rows.append({"query": q if rank == 1 else "", "rank": rank, "WORD-level": w, "CHAR-level": c})
pd.DataFrame(rows)

,query,rank,WORD-level,CHAR-level
0,تربية الاطفال,1,كيف تعلمين طفلك الكلام (0.26),كيف تعلمين طفلك الكلام (0.33)
1,,2,دفتر يوميات المشاغبين (0.21),u200e10 خطوات لتربية شبه مثالية للأبناءu200e ...
2,,3,هكذا... تعتنين بطفلك منذ الولادة وحتى سن الخا ...,دفتر يوميات المشاغبين (0.23)
3,الحرب العالمية الثانية,1,مشاهير قادة الحرب العالمية الثانية : جياب (0.27),مشاهير قادة الحرب العالمية الثانية : جياب (0.20)
4,,2,من الحرب إلى سياسة الأحزاب (0.24),من الحرب إلى سياسة الأحزاب (0.20)
5,,3,معايير التدقيق الداخلي الدولية للحد من الغش و ...,كوفي أنان : رجل سلام في عالم من الحروب (0.15)
6,الذكاء الاصطناعي,1,وصف أقاليم شبه الجزيرة العربية (0.00),اقتصاديات الإنتاج الصناعي (0.12)
7,,2,نصوص مختارة من كنيسة المشرق القرون 11-14 (0.00),ما بعد النفط (0.10)
8,,3,مدخل إلى أنظمة إلكترونيات الطيران (0.00),أبناء البحر (0.08)
9,هاري بوتر,1,وصف أقاليم شبه الجزيرة العربية (0.00),التسويق الالكتروني - طريقك للثراء عبر الانترن ...


In [5]:
# Step 5 — two lessons from the table above
# (a) TRAP: "الذكاء الاصطناعي" and "هاري بوتر" are NOT in our catalog. Word-level scores are 0.00,
#     but char-level still returns books (~0.1). Search never says "nothing found" — the AGENT must.
# (b) DATA BUG: a title shows the literal text "u200e" — an invisible direction mark the scraper saved as text.
from bookrec.prepare_data import load_raw

pattern = r"u200[ef]|u202[a-e]|[\u200e\u200f\u202a-\u202e]"

def artifact_counts(df):
    hits = {c: int(df[c].astype(str).str.contains(pattern).sum()) for c in df.columns}
    return {c: n for c, n in hits.items() if n}

raw = load_raw()  # same Kaggle download the project script uses
print("raw dataset cells with the artifact:", artifact_counts(raw))
print("catalog cells with the artifact:    ", artifact_counts(engine.books))
engine.books[engine.books.title.str.contains(pattern) | engine.books.publisher.str.contains(pattern)][["book_id", "title", "publisher"]]

raw dataset cells with the artifact: {'Title': 32, 'Author': 10, 'Description': 11, 'Publisher': 63, 'Cover': 2}
catalog cells with the artifact:     {'title': 2, 'publisher': 2}


,book_id,title,publisher
11,B012,تجليات جبل,ببلومانيا للنشر والتوزيعu200e
60,B061,u200eكيف تجني مليوناً ببطءu200e,منشورات الحلبي الحقوقية
87,B088,البروتوكولات والمراسم في عهد صاحب العظمة الشيخ...,دار الروافد الثقافية ناشرونu200e
121,B122,u200e10 خطوات لتربية شبه مثالية للأبناءu200e,مكتبة جرير


In [6]:
# Step 6 — after the fix (commit 8214697): pull the new code, REBUILD the catalog, check again
!cd repo && git pull -q && python -m bookrec.prepare_data && git diff --stat
import importlib, shutil
import bookrec.search
importlib.reload(bookrec.search)

engine = bookrec.search.BookSearch("repo/data/catalog.csv")
print("catalog cells with the artifact:", artifact_counts(engine.books))
shutil.copy("repo/data/catalog.csv", "/kaggle/working/catalog.csv")  # downloadable from the Output panel
pd.DataFrame(engine.search("تربية الاطفال"))[["score", "book_id", "title", "author", "category"]]

raw: 8986 books -> catalog: 200 books (10 categories x 20)
fingerprint: 3d092a3bb7
saved to data/catalog.csv
 data/catalog.csv | 100 +++++++++++++++++++++++++++----------------------------
 1 file changed, 50 insertions(+), 50 deletions(-)
catalog cells with the artifact: {}


,score,book_id,title,author,category
0,0.331,B138,كيف تعلمين طفلك الكلام,فرنسوا إيتيان,الأسرة والطفل
1,0.279,B135,صورة الطفولة في التربية الإسلامية,إبراهيم الخطيب ، أحمد الزبادي,الأسرة والطفل
2,0.208,B125,الصيادان,أسماء السويدي,الأسرة والطفل
3,0.203,B124,الرياضيات السريعة للأطفال,بيل هاندلي,الأسرة والطفل
4,0.199,B139,هكذا... تعتنين بطفلك منذ الولادة وحتى سن الخامسة,أمين بركات,الأسرة والطفل


In [7]:
# Step 7 — this list of dicts is EXACTLY what the agent's search tool will return in Milestone 3
engine.search("رواية تاريخية", k=3)

[{'book_id': 'B192',
  'title': 'تاريخ العرب القديم',
  'author': 'توفيق برو',
  'category': 'التاريخ والجغرافيا',
  'year': 2011,
  'description': 'يؤرخ الجزيرة العربية من كل الجوانب السياسية والحضارية (الاقتصاد والاجتماعية والفكرية والدينية) مع دراسة للمصادر التاريخية الخاصة بالمنطقة، والنظري ات المتعلقة بصلة العرب بالساميين فضلاً عن تناوله جغرافية المنطقة وأنساب العرب وأيامهم. عرضات المتعلقة بصلة العرب بالساميين فضلاً عن تناوله جغرافية المنطق',
  'score': 0.177},
 {'book_id': 'B012',
  'title': 'تجليات جبل',
  'author': 'قادة جليد',
  'category': 'الأدب والخيال',
  'year': 2018,
  'description': 'رواية من التراث الجزائري تتحدث عن العادات والتقاليد والرموز التي لا يمكن نسيانها او تجاوز سطوتها وسيطرتها على الناس عرض',
  'score': 0.163},
 {'book_id': 'B168',
  'title': 'المسرح في لبنان الشمالي منذ نشأته حتى نهاية القرن العشرين',
  'author': 'نزيه كبارة',
  'category': 'الفنون',
  'year': None,
  'description': 'سيجد القارئ أن كتاب الروايات التمثيلية استوحوا، بالإجمال، أحداث التاريخ الع